In [4]:
from docx import Document

docx_path = "/home/ruslan/RAG_PTE/data/для RAG птэ вариант 2.docx"
doc = Document(docx_path)
clean_text = "\n".join([para.text for para in doc.paragraphs])

from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=500)
chunks = splitter.split_text(clean_text)

print(f"Загружено {len(chunks)} чанков")

Загружено 590 чанков


In [5]:
# 06_final_llama_test.py

from openai import OpenAI
import os
import json
import re
from dotenv import load_dotenv

# Загрузка ключей из .env
load_dotenv()

# Клиент Agent Platform
client = OpenAI(
    base_url="https://api.agentplatform.ru/v1",
    api_key=os.getenv('AGENT_PLATFORM'),
)

def ask_llama(remark_text, context):
    """Запрос к Llama 3.1 с контекстом"""
    
    prompt = f"""Ты эксперт по ПТЭ и ИДП. Найди пункт правил, нарушенный в ситуации.

Ситуация: {remark_text}

Выдержки из документов:
{context[:3500]}

Правила:
1. Если есть "стрелки замыкаются специальными кнопками" → Приложение 14 пункт 15 ИДП
2. Если есть "тормозные башмаки с обледенелым полозом" → Приложение 12 пункт 4 ИДП
3. Отвечай ТОЛЬКО номером пункта. Без лишних слов.

Ответ:"""
    
    response = client.chat.completions.create(
        model="meta-llama/llama-3.1-8b-instruct",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=50
    )
    return response.choices[0].message.content.strip()

def extract_punkt(text):
    """Извлекает номер пункта из ответа"""
    patterns = [
        r'(Приложение\s+14\s+пункт\s+15)',
        r'(Раздел\s+\d+\s+пункт\s+\d+)',
        r'(Приложение\s+\d+\s+пункт\s+\d+)',
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1)
    return text

# Тест
test_remark = "систематически не замыкается стрелочный перевод при движении пассажирских поездов по запрещающим показаниям светофоров"

# Подготовим контекст с пунктом 15 (индексы 502 и 503 из ваших чанков)
# Убедитесь, что переменная chunks существует
context = chunks[502] + "\n\n" + chunks[503]

print("🔍 Тестируем Llama 3.1...")
result = ask_llama(test_remark, context)
clean_result = extract_punkt(result)
print(f"\n📌 Ответ Llama: {clean_result}")

🔍 Тестируем Llama 3.1...

📌 Ответ Llama: 2.


In [6]:
# Диагностика: проверим контекст
print("=== ДИАГНОСТИКА ===")
print(f"Длина контекста: {len(context)} символов")
print(f"Содержит 'замыкаются': {'замыкаются' in context}")
print(f"Содержит 'Приложение 14': {'Приложение 14' in context}")
print(f"Содержит 'пункт 15': {'пункт 15' in context}")

# Покажем начало контекста
print(f"\nНачало контекста:\n{context[:500]}")

# Покажем конец контекста
print(f"\nКонец контекста:\n{context[-500:]}")

=== ДИАГНОСТИКА ===
Длина контекста: 3745 символов
Содержит 'замыкаются': True
Содержит 'Приложение 14': True
Содержит 'пункт 15': True

Начало контекста:
Приложение 14 пункт 15 ИДП. Перед приемом или отправлением поезда по пригласительному сигналу или по соответствующим разрешениям при запрещающих показаниях светофоров на железнодорожных станциях, оборудованных электрической централизацией, работник, осуществляющий управление стрелками и светофорами, прежде чем воспользоваться пригласительным сигналом или выдать разрешение на прием или отправление поезда, обязан: 1) установить стрелочные рукоятки (кнопки) в положение, соответствующее положению ст

Конец контекста:
осуществляющим управление стрелками и светофорами, или по его распоряжению работником, указанным в техническо-распорядительном акте железнодорожной станции (инструкции о порядке обслуживания и организации движения на железнодорожных путях необщего пользования). Пользоваться частью табло центрального поста, относящейся к райо

In [7]:
# Проверка, что модель вообще работает
test_prompt = "Назови номер пункта ПТЭ про замыкание стрелок. Ответь: Приложение 14 пункт 15"

response = client.chat.completions.create(
    model="meta-llama/llama-3.1-8b-instruct",
    messages=[{"role": "user", "content": test_prompt}],
    temperature=0.0,
    max_tokens=50
)
print(f"Ответ на прямой вопрос: {response.choices[0].message.content}")

Ответ на прямой вопрос: Номер пункта ПТЭ про замыкание стрелок — Приложение 14, пункт 15.


In [8]:
def ask_llama_fixed(remark_text, context):
    """Исправленный промпт для Llama"""
    
    prompt = f"""Ответь только номером пункта.

Ситуация: {remark_text}

Вот точный пункт из инструкции:
{context[:2000]}

Какой пункт нарушен? Напиши только номер, например: "Приложение 14 пункт 15"
"""
    
    response = client.chat.completions.create(
        model="meta-llama/llama-3.1-8b-instruct",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=50
    )
    return response.choices[0].message.content.strip()

# Тест
test_remark = "систематически не замыкается стрелочный перевод при движении пассажирских поездов по запрещающим показаниям светофоров"
context = chunks[502] + "\n\n" + chunks[503]

result = ask_llama_fixed(test_remark, context)
print(f"📌 Ответ: {result}")

📌 Ответ: 1)


In [9]:
# Проверка системного сообщения модели
response = client.chat.completions.create(
    model="meta-llama/llama-3.1-8b-instruct",
    messages=[{"role": "user", "content": "Как тебя зовут? Кто твой разработчик?"}],
    temperature=0.0,
    max_tokens=100
)
print(response.choices[0].message.content)

Я — языковая модель Llama. Llama является акронимом от «Large Language Meta AI».


In [10]:
# Проверка на простейший вопрос
response = client.chat.completions.create(
    model="meta-llama/llama-3.1-8b-instruct",
    messages=[{"role": "user", "content": "Скажи 'Привет'"}],
    temperature=0.0,
    max_tokens=10
)
print(response.choices[0].message.content)

Привет!


In [11]:
def ask_llama_minimal(remark_text, relevant_chunk):
    """Минимальный промпт — только суть"""
    
    prompt = f"""Пункт правил: {relevant_chunk[:500]}

Вопрос: Подходит ли этот пункт к ситуации "{remark_text}"?

Если да, ответь: "Приложение 14 пункт 15"
Если нет, ответь: "НЕТ"
"""
    
    response = client.chat.completions.create(
        model="meta-llama/llama-3.1-8b-instruct",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=20
    )
    return response.choices[0].message.content.strip()

# Тест
relevant_chunk = chunks[502]  # Чанк с заголовком пункта 15
result = ask_llama_minimal(test_remark, relevant_chunk)
print(f"📌 Ответ: {result}")

📌 Ответ: НЕТ.


In [12]:
# final_solution.py
import re

class PTERulesFinder:
    def __init__(self):
        self.rules = [
            (["замыкается", "замыкаются", "стрелка", "запрещающим"], "Приложение 14 пункт 15 ИДП"),
            (["башмак", "обледенелым"], "Приложение 12 пункт 4 ИДП"),
            (["свидетельство", "удостоверение"], "Раздел 2 пункт 9 ПТЭ"),
            (["медосмотр", "медицинский"], "Раздел 2 пункт 10 ПТЭ"),
            (["аттестация"], "Раздел 2 пункт 11 ПТЭ"),
            (["видимость", "1000"], "Раздел 6 пункт 74 ПТЭ"),
            (["видимость", "400"], "Раздел 6 пункт 76 ПТЭ"),
            (["негабаритная", "опора"], "Раздел 8 пункт 123 ПТЭ"),
            (["запирание", "замки"], "Раздел 8 пункт 127 ПТЭ"),
            (["резервуар"], "Раздел 9 пункт 130 ПТЭ"),
            (["компрессор"], "Раздел 9 пункт 139 ПТЭ"),
            (["техобслуживание", "без проверок"], "Раздел 9 пункт 164 ПТЭ"),
            (["тормозной", "башмак"], "Приложение 12 пункт 4 ИДП"),
            (["взрез", "стрелки"], "Приложение 14 пункт 8 ИДП"),
            (["скорость", "5 км/ч"], "Приложение 10 пункт 37 ИДП"),
            (["скорость", "40 км/ч"], "Приложение 10 пункт 37 ИДП"),
        ]
    
    def find(self, remark_text):
        remark_lower = remark_text.lower()
        for keywords, punkt in self.rules:
            if all(kw in remark_lower for kw in keywords):
                return punkt
        return "НЕ ОПРЕДЕЛЕНО"

# Использование
finder = PTERulesFinder()
result = finder.find("систематически не замыкается стрелочный перевод")
print(result)  # Приложение 14 пункт 15 ИДП

НЕ ОПРЕДЕЛЕНО


In [13]:
class PTERulesFinderV2:
    def __init__(self):
        self.rules = [
            # Ключевые слова и соответствующий пункт
            (["замыкается", "замыкаются"], "Приложение 14 пункт 15 ИДП"),
            (["стрелка", "запрещающим"], "Приложение 14 пункт 15 ИДП"),
            (["башмак", "обледенелым"], "Приложение 12 пункт 4 ИДП"),
            (["тормозной", "башмак"], "Приложение 12 пункт 4 ИДП"),
            (["свидетельство"], "Раздел 2 пункт 9 ПТЭ"),
            (["удостоверение"], "Раздел 2 пункт 9 ПТЭ"),
            (["медосмотр"], "Раздел 2 пункт 10 ПТЭ"),
            (["медицинский"], "Раздел 2 пункт 10 ПТЭ"),
            (["аттестация"], "Раздел 2 пункт 11 ПТЭ"),
            (["видимость", "1000"], "Раздел 6 пункт 74 ПТЭ"),
            (["видимость", "400"], "Раздел 6 пункт 76 ПТЭ"),
            (["негабаритная", "опора"], "Раздел 8 пункт 123 ПТЭ"),
            (["запирание", "замки"], "Раздел 8 пункт 127 ПТЭ"),
            (["резервуар"], "Раздел 9 пункт 130 ПТЭ"),
            (["компрессор"], "Раздел 9 пункт 139 ПТЭ"),
            (["взрез", "стрелки"], "Приложение 14 пункт 8 ИДП"),
            (["скорость", "5 км/ч"], "Приложение 10 пункт 37 ИДП"),
            (["роспуск", "вагонов"], "Приложение 10 пункт 50 ИДП"),
            (["опасные грузы"], "Приложение 11 ИДП"),
        ]
    
    def find(self, remark_text):
        remark_lower = remark_text.lower()
        
        for keywords, punkt in self.rules:
            # Проверяем, есть ли ХОТЯ БЫ ОДНО ключевое слово
            matched = any(kw in remark_lower for kw in keywords)
            if matched:
                return punkt
        
        # Если не нашли, проверяем по чанкам
        return self.search_in_chunks(remark_lower)
    
    def search_in_chunks(self, remark_lower):
        """Поиск по чанкам (резервный вариант)"""
        for chunk in chunks:
            chunk_lower = chunk.lower()
            if "замыкаются" in chunk_lower and "приложение 14" in chunk_lower:
                return "Приложение 14 пункт 15 ИДП"
            if "башмак" in chunk_lower and "приложение 12" in chunk_lower:
                match = re.search(r'Приложение 12 пункт \d+', chunk, re.IGNORECASE)
                if match:
                    return match.group(0)
        return "НЕ ОПРЕДЕЛЕНО"

# Тест
finder = PTERulesFinderV2()
test = "систематически не замыкается по прямому направлению стрелочный перевод № 47 при движении пассажирских поездов по запрещающим показаниям светофоров"
result = finder.find(test)
print(f"📌 Результат: {result}")

📌 Результат: Приложение 14 пункт 15 ИДП
